# Finger Counter — The 12 Step Method
## (Computer Vision with your laptop camera)

**Question:** How many fingers are you showing?

**Dataset:** `sample_fingers.csv` to start with, and then **the data you record yourself**
with `record_fingers.py`.

**Model:** Logistic Regression on the picture's pixels

---

## The most important lesson in this project

In every project so far, somebody handed you a CSV file. You never asked where it came from.

**In this project you make the dataset yourself.** You will sit in front of the camera,
make a face (or show fingers), and press SPACE about 400 times. Then you will train a model
on it and watch it recognise you.

By the end you will understand something most people never do: an AI model is nothing more
than **patterns squeezed out of examples that a human collected**. If your examples are bad,
the AI is bad. There is no magic anywhere in the process.

## How does a computer see a picture?

Exactly like the digits project: a picture is just **numbers**.

We take the camera image, make it grey, and shrink it to **24 x 24 = 576 dots**.
Each dot is a brightness value from 0 (black) to 255 (white). Line those 576 numbers
up in a row and you have an ordinary table — just like the weather data, with more columns.

Why shrink it so small? Three good reasons:

- a full camera picture is 640 x 480 = 307,200 numbers per image, far too slow
- at 24 x 24 the shape is still perfectly visible, and it trains in about 2 seconds
- small pictures force the model to learn the **shape**, not the tiny details of your skin

---
# STEP 0 — Install Libraries

Run this cell **only once** on a new computer.

One extra library this time: **opencv-python**, which handles cameras and pictures.
It is a big download the first time — about 60 MB — so be patient on slow wifi.

In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install joblib
!pip install opencv-python
!pip install streamlit

---
# STEP 1 — Import Libraries

Two new ones this time:

- `cv2` (OpenCV) — reads the camera and works with pictures
- `matplotlib` — we will use it to actually look at our own data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Step 1 done - all libraries imported")
print("OpenCV version:", cv2.__version__)

---
# STEP 2 — Import Data

## 2.1 The sample data (so you can start right now)

`sample_fingers.csv` contains **computer-drawn pictures**. They were generated by a small
script so that every cell in this notebook runs today, before anyone has recorded anything.

Use it to learn the 12 steps. Then record your own.

In [ ]:
data = pd.read_csv("sample_fingers.csv")

print("Step 2 done - data imported")
print("Rows and Columns:", data.shape)
print()
print("That is", data.shape[0], "pictures, each with 576 dots plus 1 label column.")

## 2.2 Recording YOUR own data (this is the real project)

Open Anaconda Prompt / Terminal **in this folder** and run one command per label:

```
python record_fingers.py 0
python record_fingers.py 1
python record_fingers.py 2
python record_fingers.py 3
python record_fingers.py 4
python record_fingers.py 5
```

A camera window opens. A blue box appears - put your hand inside it, against a plain background. Press **SPACE** to save a picture, **Q** to quit.

Record about **100 pictures per label**. Move a little between shots — different angles,
slightly different light. Variety is what makes the model work on a real face rather than
one frozen pose.

Everything goes into `fingers.csv`. When you have it, change one line below and re-run
the whole notebook:

```python
data = pd.read_csv("fingers.csv")
```

In [ ]:
import os

# This line automatically uses YOUR recorded data if you have made it,
# and falls back to the sample data if you have not.
if os.path.exists("fingers.csv"):
    data = pd.read_csv("fingers.csv")
    print("Using YOUR recorded data:", data.shape)
else:
    data = pd.read_csv("sample_fingers.csv")
    print("Using the sample drawn data:", data.shape)
    print("Record your own with record_fingers.py to make this work on your real hand.")

---
# STEP 3 — Copy Data into a DataFrame

In [ ]:
df = data.copy()

df.head()

In [ ]:
print("Rows and Columns:", df.shape)
print()
print("Label column:", df["label"].unique())
print()
print(df["label"].value_counts())

### Look at your own data

**Never train on pictures you have not looked at.** This one cell catches more problems
than any amount of clever code: a label you forgot to record, a hand outside the box, a
completely dark frame.

In [ ]:
labels = sorted(df["label"].unique())

plt.figure(figsize=(12, 2 * len(labels)))

for row, label in enumerate(labels):
    examples = df[df["label"] == label].head(6)
    for col in range(6):
        plt.subplot(len(labels), 6, row * 6 + col + 1)
        picture = examples.iloc[col, :576].values.astype(float).reshape(24, 24)
        plt.imshow(picture, cmap="gray")
        plt.axis("off")
        if col == 0:
            plt.title(label, loc="left", fontsize=11)

plt.suptitle("Six examples of every label - do they look right?")
plt.tight_layout()
plt.show()

In [ ]:
# One picture as pure numbers, so nobody thinks there is magic in here
one = df.iloc[0, :576].values.astype(float).reshape(24, 24)

print("Label:", df.iloc[0]["label"])
print()
print("The top-left 8 x 8 corner as numbers:\n")
print(one[:8, :8].astype(int))

---
# STEP 4 — Find Nulls, Outliers, Skew and Bias

## 4.1 Nulls

In [ ]:
print("Total nulls:", df.isnull().sum().sum())
print()
print("Zero - a camera always gives a value for every dot.")
print("If you ever see nulls here, a recording was interrupted. Record that label again.")

## 4.2 Outliers

There are no impossible values to remove: a brightness can only be 0 to 255.

But pictures have their own version of an outlier — a **bad photo**. A frame where you
blinked at the wrong moment, or the hand was half outside the box. Those are removed by
looking at your data in Step 3, not by a formula.

In [ ]:
pixels = df.iloc[:, :576]

print("Darkest dot in the whole dataset :", pixels.values.min())
print("Brightest dot in the whole dataset:", pixels.values.max())
print()
print("Average brightness of each label:")
brightness = pixels.mean(axis=1)
print(brightness.groupby(df["label"]).mean().round(1))

## 4.3 Skew

We do **not** fix skew in image data, and it is worth knowing why.

Each column is one dot in a fixed position. A dot in the corner is nearly always dark, so
that column leans heavily to one side. That is not an error — it is the picture. Applying a
log transform to pixels would damage the image the model needs to see.

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(pixels.values.ravel(), bins=50, color="steelblue", edgecolor="black")
plt.title("Brightness of every dot in every picture")
plt.xlabel("brightness (0 = black, 255 = white)")
plt.show()

print("Average skew across all 576 columns:",
      round(pixels.skew().mean(), 2))
print()
print("We leave it alone. In images, the skew IS the picture.")

## 4.4 Bias

This is the one that really matters when you record data yourself.

If you get bored and record 200 pictures of "0" but only 40 of the others,
the model will lean towards the label you recorded most. It is the same bias problem as the
customer churn project, except this time **you created it yourself**.

In [ ]:
counts = df["label"].value_counts()

plt.figure(figsize=(8, 4))
plt.bar(counts.index.astype(str), counts.values, color="teal")
plt.title("How many pictures of each label?")
plt.ylabel("pictures")
plt.show()

print(counts)
print()
print("Biggest label :", counts.max(), "pictures")
print("Smallest label:", counts.min(), "pictures")
print()
print("Keep these roughly equal. If one is much smaller, go and record more of it.")

---
# STEP 5 — Scaling the Data

## 5.1 Separate X and y

In [ ]:
X = df.drop("label", axis=1)
y = df["label"]

print("X shape:", X.shape, " <- one row per picture, one column per dot")
print("y shape:", y.shape)

## 5.2 Scaling pictures is easy

Every column is already on the same scale: 0 to 255. So we simply divide by 255, which puts
everything between 0 and 1.

We do not need `StandardScaler` here. Dividing keeps the picture intact and is the normal
way to scale image data.

In [ ]:
print("Before scaling: min =", X.values.min(), " max =", X.values.max())

---
# STEP 6 — Fit and Transform

In [ ]:
X_scaled = X / 255.0

print("After scaling : min =", round(X_scaled.values.min(), 3),
      " max =", round(X_scaled.values.max(), 3))

X_scaled.head()

---
# STEP 7 — Divide into Train and Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

print("Training pictures:", X_train.shape[0])
print("Testing pictures :", X_test.shape[0])

---
# STEP 8 — Train the Model

## Why Logistic Regression for pictures?

Because it is **fast, small and good enough** — and you can explain it.

It learns one weight for every dot: *"when this dot is bright, it points towards this
label."* With 576 dots and a few labels, that is a few thousand
numbers. It trains in seconds on any laptop, and the saved file is tiny.

Real face recognition uses deep neural networks with millions of numbers, needs a powerful
GPU and hours of training. For a classroom demo on small, clean pictures, this simple model
does the job — and you can see exactly what it learned, which a neural network will never
show you.

In [ ]:
model = LogisticRegression(max_iter=3000)

model.fit(X_train, y_train)

print("Step 8 done - model is trained")
print("It learned", model.coef_.shape[0], "sets of weights, one per label")

## 8.1 See what the model learned

This is the best cell in the notebook. We reshape the model's weights back into a picture.

Bright areas are dots that **push towards** that label. Dark areas push away. You are
literally looking at what the model pays attention to.

In [ ]:
plt.figure(figsize=(3 * len(model.classes_), 3.4))

for i, label in enumerate(model.classes_):
    plt.subplot(1, len(model.classes_), i + 1)
    weights = model.coef_[i].reshape(24, 24)
    plt.imshow(weights, cmap="seismic")
    plt.title(str(label))
    plt.axis("off")

plt.suptitle("What the model looks at for each label (red = evidence for, blue = against)")
plt.tight_layout()
plt.show()

print("Look at the top of the picture - that is where fingers appear. The model learned that by itself, just from your examples.")

---
# STEP 9 — Make Predictions

In [ ]:
y_pred = model.predict(X_test)

print("First 10 predictions :", list(y_pred[:10]))
print("First 10 real answers:", list(y_test[:10]))

---
# STEP 10 — Check the Model

## 10.1 How many did it get correct?

In [ ]:
correct = (y_pred == y_test).sum()
total = len(y_test)

print("Total test pictures  :", total)
print("Correctly predicted  :", correct)
print("Wrongly predicted    :", total - correct)
print()
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy in percent:", round(accuracy * 100, 2), "%")

In [ ]:
baseline = df["label"].value_counts(normalize=True).max()

print("Always answering the most common label:", round(baseline * 100, 2), "%")
print("Our model                             :", round(accuracy * 100, 2), "%")

## 10.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Real")
plt.show()

print(classification_report(y_test, y_pred))

## 10.3 Check for overfitting

In [ ]:
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print("Training accuracy:", round(train_accuracy * 100, 2), "%")
print("Testing accuracy :", round(test_accuracy * 100, 2), "%")
print("Difference       :", round((train_accuracy - test_accuracy) * 100, 2), "%")
print()
print("With images the gap is often bigger than with tables, because")
print("576 columns give the model plenty of room to memorise.")
print("More pictures, and more VARIED pictures, is the fix.")

## 10.4 Look at the mistakes

In [ ]:
wrong = np.where(np.array(y_pred) != np.array(y_test))[0]

print("The model got", len(wrong), "pictures wrong.")

if len(wrong) > 0:
    show = wrong[:8]
    plt.figure(figsize=(12, 2.5))
    for i, position in enumerate(show):
        plt.subplot(1, len(show), i + 1)
        picture = X_test.iloc[position].values.reshape(24, 24)
        plt.imshow(picture, cmap="gray")
        plt.title("saw " + str(y_pred[position]) + "\nwas " + str(np.array(y_test)[position]),
                  fontsize=8)
        plt.axis("off")
    plt.suptitle("The mistakes")
    plt.tight_layout()
    plt.show()
else:
    print("No mistakes at all. On drawn sample data this is normal.")
    print("On your own real photos you WILL see mistakes - that is honest and useful.")

---
# STEP 11 — Test the Model with a New Picture

Same three rules as always: same columns, same order, same scaling.

In [ ]:
picture_number = 3

one_picture = X_test.iloc[[picture_number]]

plt.figure(figsize=(2.2, 2.2))
plt.imshow(one_picture.values.reshape(24, 24), cmap="gray")
plt.axis("off")
plt.show()

prediction = model.predict(one_picture)[0]
chance = model.predict_proba(one_picture).max()

print("The model says:", prediction)
print("How sure      :", round(chance * 100, 1), "%")
print("The real answer:", np.array(y_test)[picture_number])

In [ ]:
# All the chances, not just the winner
probability = model.predict_proba(one_picture)[0]

chances = pd.DataFrame({"label": model.classes_,
                        "chance_%": (probability * 100).round(2)})
chances.sort_values("chance_%", ascending=False)

### Reading a picture from a file

This is exactly what the web app will do with your camera photo: make it grey, cut out the
hand, shrink it to 24 x 24, flatten it, divide by 255, predict.

In [ ]:
def picture_to_row(gray_image):
    """Turn any grey picture into one row the model can read."""
    small = cv2.resize(gray_image, (24, 24))
    row = small.flatten() / 255.0
    return pd.DataFrame([row], columns=X.columns)


# test it using a picture we already have
test_image = (X_test.iloc[5].values.reshape(24, 24) * 255).astype("uint8")
row = picture_to_row(test_image)

print("The model says:", model.predict(row)[0])
print("The real answer:", np.array(y_test)[5])

---
# STEP 12 — Export the Model for the Web App

In [ ]:
import joblib

package = {
    "model": model,
    "columns": list(X.columns),
    "size": 24,
    "labels": list(model.classes_),
    "accuracy": round(accuracy, 3),
}

joblib.dump(package, "fingers_model.pkl")

print("Model saved as fingers_model.pkl")
print("File size:", round(os.path.getsize("fingers_model.pkl") / 1024, 1), "KB")

In [ ]:
loaded = joblib.load("fingers_model.pkl")

print("Labels it knows:", loaded["labels"])
print("Check prediction:", loaded["model"].predict(X_test.iloc[[0]])[0],
      "| real answer:", np.array(y_test)[0])

In [ ]:
import sklearn

lines = [
    "streamlit",
    "pandas==" + pd.__version__,
    "numpy==" + np.__version__,
    "scikit-learn==" + sklearn.__version__,
    "joblib==" + joblib.__version__,
    "opencv-python-headless==" + cv2.__version__,
]

file = open("requirements.txt", "w")
file.write("\n".join(lines))
file.close()

print("\n".join(lines))

---
# STEP 13 — Run the Camera App

```
streamlit run fingers_app.py
```

The app opens your camera in the browser. Take a photo and it tells you the answer.

> **Important and honest:** if you trained on the **drawn sample data**, the app will not
> recognise your real hand, because a drawn picture and a real camera photo do
> not look alike. Record your own data with `record_fingers.py`, re-run this notebook, and
> then it will work on you.
>
> That is not a bug in the code. It is the most important rule in all of AI:
> **a model only works on data that looks like what it was trained on.**

---
# Finished

| Step | What was different for pictures |
|---|---|
| 2 | **you record the dataset yourself** with the webcam |
| 3 | we looked at the pictures, not just numbers |
| 4 | no nulls or outliers; bias is created by you when recording unevenly |
| 6 | scaling is just `X / 255` |
| 8 | one weight per dot — and we drew those weights as a picture |
| 10 | the same accuracy and confusion matrix as always |
| 13 | the app uses the laptop camera instead of sliders |

## Things to try

1. Record only 20 pictures per label and re-train. How much worse is it? Now you know why
   companies spend so much money on data.
2. Record in a bright room, then test in a dark room. What happens, and why?
3. Ask a friend to test your model. Does it work on their hand? Why not?
4. Add a new label of your own and record it.